In [1]:
%cd ../

/u02/thanh/workplace/plasma


In [2]:
import inspect
import plasma.data_model as pdm

from pydantic import BaseModel
from typing import NamedTuple, dataclass_transform

from plasma.data_model.base_model.field import Field, Composite, List

In [3]:
@pdm.model
class A:
    a:int
    b:int
    c:list[str]

In [4]:
@pdm.model
class B:
    b1:A
    b2:list[A]
    b3:list

In [5]:
@pdm.model
class C:
    c1:B
    c2:list[B]

In [6]:
c = C(
        B(
            A(5, 6, ['a', 'b']),
            [
                A(5, 6, ['a', 'b']),
            ],
            [1, 2]
        ),
        [
            B(
                A(5, 6, ['a', 'b']),
                [],
                [1, 3]
            ),
            B(
                A(5, 6, ['a', 'b']),
                [
                    A(7, 8, ['a', 'b']),
                ],
                [1, 3]
            ),
        ]
    )

c

C
  |-c1=B
    |-b1=A
      |-a=5
      |-b=6
      |-c=['a', 'b']
    |-b2:
      A
        |-a=5
        |-b=6
        |-c=['a', 'b']
    |-b3=[1, 2]
  |-c2:
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2=[]
      |-b3=[1, 3]
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2:
        A
          |-a=7
          |-b=8
          |-c=['a', 'b']
      |-b3=[1, 3]

In [8]:
import re

In [10]:
class StructState(dict):
    
    def __init__(self, obj):
        super().__init__()

        if obj is not None:
            struct = pdm.type_inquirer.struct(type(obj))
            for k in struct:
                self.__update(struct, k, obj)
        
    def __update(self, template, key, obj):
        next_template = template[key]
        value = getattr(obj, key)
        if isinstance(next_template, dict):
            self[key] = StructState(value)
        elif isinstance(next_template, list):
            assert isinstance(value, (list, tuple)), f'expected list or tuple for {obj} at {key}'
            self[key] = [StructState(v) for v in value]
        else:
            self[key] = value
            
sc = StructState(c)
sc

{'c1': {'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']},
  'b2': [{'a': 5, 'b': 6, 'c': ['a', 'b']}],
  'b3': [1, 2]},
 'c2': [{'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']}, 'b2': [], 'b3': [1, 3]},
  {'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']},
   'b2': [{'a': 7, 'b': 8, 'c': ['a', 'b']}],
   'b3': [1, 3]}]}

In [11]:
class struct2accessor(dict):
    
    def __init__(self, schema:dict, struct:dict):
        super().__init__()
        
        for k, v in struct.items():
            self.__update(schema[k], k, v)    

    def __update(self, schema, key, value):
        if isinstance(schema, list):
            for i, v in enumerate(value):
                self.__update(schema[0],f'{key}.{i}', v)
        elif isinstance(schema, dict) and value is not None:
            for k, v in value.items():
                self.__update(schema[k], f'{key}.{k}', v)
        else:
            self[key] = value

In [12]:
struct2accessor(C.__struct, sc)

{'c1.b1.a': 5,
 'c1.b1.b': 6,
 'c1.b1.c': ['a', 'b'],
 'c1.b2.0.a': 5,
 'c1.b2.0.b': 6,
 'c1.b2.0.c': ['a', 'b'],
 'c1.b3': [1, 2],
 'c2.0.b1.a': 5,
 'c2.0.b1.b': 6,
 'c2.0.b1.c': ['a', 'b'],
 'c2.0.b3': [1, 3],
 'c2.1.b1.a': 5,
 'c2.1.b1.b': 6,
 'c2.1.b1.c': ['a', 'b'],
 'c2.1.b2.0.a': 7,
 'c2.1.b2.0.b': 8,
 'c2.1.b2.0.c': ['a', 'b'],
 'c2.1.b3': [1, 3]}

In [13]:
ac

{'c1.b1.a': 5,
 'c1.b1.b': 6,
 'c1.b1.c': ['a', 'b'],
 'c1.b2.0.a': 5,
 'c1.b2.0.b': 6,
 'c1.b2.0.c': ['a', 'b'],
 'c1.b3': [1, 2],
 'c2.0.b1.a': 5,
 'c2.1.b1.a': 5,
 'c2.0.b1.b': 6,
 'c2.1.b1.b': 6,
 'c2.0.b1.c': ['a', 'b'],
 'c2.1.b1.c': ['a', 'b'],
 'c2.1.b2.0.a': 7,
 'c2.1.b2.0.b': 8,
 'c2.1.b2.0.c': ['a', 'b'],
 'c2.0.b3': [1, 3],
 'c2.1.b3': [1, 3]}